In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_pickle("../data/raw/AT.pkl")

In [3]:
# Inspect columns
print("Columns in the dataset:")
for col in df.columns:
    print(f"- {col}")

Columns in the dataset:
- Unnamed: 0
- Company name Latin alphabet
- Country ISO code
- City
Latin Alphabet
- NACE Rev. 2, core code (4 digits)
- BvD ID number
- NACE Rev. 2 main section
- Region in country
- Status
- Date of incorporation
- Number of employees
2024
- Number of employees
2023
- Number of employees
2022
- Number of employees
2021
- Number of employees
2020
- Number of employees
2019
- Number of employees
2018
- Number of employees
2017
- Founded Year
- Region in country clean


In [4]:
# Inspect employee columns carefully
employee_cols = [col for col in df.columns if 'Number of employees' in col]
print("Employee columns:")
for col in employee_cols:
    print(f"- {col}")

print(f"\nNumber of employee columns: {len(employee_cols)}")

# Check unique values in each employee column to understand missing value patterns
for col in employee_cols:
    unique_vals = df[col].unique()
    print(f"\n{col} - Unique values (first 10): {unique_vals[:10]}")
    print(f"  Total unique: {len(unique_vals)}")
    print(f"  Null count: {df[col].isnull().sum()}")
    print(f"  'n.a.' count: {(df[col] == 'n.a.').sum()}")
    print(f"  Blank string count: {(df[col] == '').sum()}")

Employee columns:
- Number of employees
2024
- Number of employees
2023
- Number of employees
2022
- Number of employees
2021
- Number of employees
2020
- Number of employees
2019
- Number of employees
2018
- Number of employees
2017

Number of employee columns: 8

Number of employees
2024 - Unique values (first 10): ['n.a.' 23557 121 78174 49298 19224 45000 188 42564 30003]
  Total unique: 925
  Null count: 0
  'n.a.' count: 15013
  Blank string count: 0

Number of employees
2023 - Unique values (first 10): ['n.a.' 20592 108 77136 50595 17137 43061 185 44887 29717]
  Total unique: 943
  Null count: 0
  'n.a.' count: 10218
  Blank string count: 0

Number of employees
2022 - Unique values (first 10): [15900 22308 108 73740 49633 'n.a.' 14895 42603 180 44414]
  Total unique: 958
  Null count: 0
  'n.a.' count: 10011
  Blank string count: 0

Number of employees
2021 - Unique values (first 10): [16200 22434 148 73606 48307 'n.a.' 41898 165 46185 26804]
  Total unique: 934
  Null count: 0
 

## Data Inspection Summary

### Which years are available?
Employee data is available for 8 years: 2017 through 2024.

### How unavailable values are stored?
- Missing values are stored as the string `"n.a."`
- No actual NaN values (pandas nulls)
- No blank strings `""`
- This matches the convention used in the Belgium notebook

### Which columns will be used for growth and classification?
**For growth analysis:**
- Employee columns: `Number of employees\n2017` through `Number of employees\n2024`
- These contain numeric values or "n.a." strings

**For classification:**
- Industry: `NACE Rev. 2, core code (4 digits)`, `NACE Rev. 2 main section`
- Location: `Region in country clean`, `City\nLatin Alphabet`
- Company status: `Status`
- Other: `Founded Year`, `Date of incorporation`

### Missing Data Patterns:
- 2017: 35,821 missing (77.8%)
- 2018: 28,196 missing (61.2%)
- 2019: 18,261 missing (39.6%)
- 2020: 20,467 missing (44.4%)
- 2021: 9,926 missing (21.5%)
- 2022: 10,011 missing (21.7%)
- 2023: 10,218 missing (22.2%)
- 2024: 15,013 missing (32.6%)

Missing data decreases over time, with more recent years having fewer missing values.

## Step 2: Data Structure Cleaning

### Objectives:
- Drop unnecessary columns
- Rename columns to simple names
- Standardize region and status labels
- Create raw and numeric versions of employee columns (preserving missingness logic)

In [5]:
# Drop unnecessary columns
columns_to_drop = ['Unnamed: 0']
df = df.drop(columns=columns_to_drop, errors='ignore')
print(f"Dropped columns: {columns_to_drop}")
print(f"Remaining columns: {len(df.columns)}")

Dropped columns: ['Unnamed: 0']
Remaining columns: 19


In [6]:
# Rename columns to simple names
column_rename_map = {
    'Company name Latin alphabet': 'company_name',
    'Country ISO code': 'country_code',
    'City\nLatin Alphabet': 'city',
    'NACE Rev. 2, core code (4 digits)': 'nace_code',
    'BvD ID number': 'bvd_id',
    'NACE Rev. 2 main section': 'nace_section',
    'Region in country': 'region_raw',
    'Status': 'status',
    'Date of incorporation': 'incorporation_date',
    'Number of employees\n2024': 'emp_2024_raw',
    'Number of employees\n2023': 'emp_2023_raw',
    'Number of employees\n2022': 'emp_2022_raw',
    'Number of employees\n2021': 'emp_2021_raw',
    'Number of employees\n2020': 'emp_2020_raw',
    'Number of employees\n2019': 'emp_2019_raw',
    'Number of employees\n2018': 'emp_2018_raw',
    'Number of employees\n2017': 'emp_2017_raw',
    'Founded Year': 'founded_year',
    'Region in country clean': 'region'
}

df = df.rename(columns=column_rename_map)
print("Columns renamed:")
for old, new in column_rename_map.items():
    print(f"  {old} -> {new}")

Columns renamed:
  Company name Latin alphabet -> company_name
  Country ISO code -> country_code
  City
Latin Alphabet -> city
  NACE Rev. 2, core code (4 digits) -> nace_code
  BvD ID number -> bvd_id
  NACE Rev. 2 main section -> nace_section
  Region in country -> region_raw
  Status -> status
  Date of incorporation -> incorporation_date
  Number of employees
2024 -> emp_2024_raw
  Number of employees
2023 -> emp_2023_raw
  Number of employees
2022 -> emp_2022_raw
  Number of employees
2021 -> emp_2021_raw
  Number of employees
2020 -> emp_2020_raw
  Number of employees
2019 -> emp_2019_raw
  Number of employees
2018 -> emp_2018_raw
  Number of employees
2017 -> emp_2017_raw
  Founded Year -> founded_year
  Region in country clean -> region


In [7]:
# Inspect region values for standardization
print("Unique region values:")
print(df['region'].value_counts().head(20))

print("\nUnique region_raw values:")
print(df['region_raw'].value_counts().head(20))

Unique region values:
region
Wien                10066
Oberosterreich       7884
Niederosterreich     7118
Steiermark           5966
Tirol                4630
Salzburg             3935
Karnten              2691
Vorarlberg           2290
Burgenland           1248
                      257
Name: count, dtype: int64

Unique region_raw values:
region_raw
Wien                10062
Oberosterreich       7863
Niederosterreich     7114
Steiermark           5941
Tirol                4624
Salzburg             3931
Karnten              2689
Vorarlberg           2270
Burgenland           1229
Name: count, dtype: int64


In [8]:
# Inspect status values for standardization
print("Unique status values:")
print(df['status'].value_counts())

Unique status values:
status
Active                             42283
Dissolved                           1529
Dissolved (liquidation)             1334
Active (insolvency proceedings)      665
Active (dormant)                     119
In liquidation                        66
Status unknown                        54
Dissolved (merger or take-over)       12
Dissolved (bankruptcy)                 8
Bankruptcy                             8
Active (default of payment)            7
Name: count, dtype: int64


In [9]:
# Create numeric versions of employee columns (preserving missingness logic)
# Following Rule 1: If input unavailable, output should be unavailable
# Raw columns keep original values, numeric columns convert valid numbers to float and "n.a." to NaN

years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

for year in years:
    raw_col = f'emp_{year}_raw'
    num_col = f'emp_{year}_num'

    # Convert to numeric, treating "n.a." as NaN
    df[num_col] = pd.to_numeric(df[raw_col], errors='coerce')

    print(f"Created {num_col}: {df[num_col].notna().sum()} valid values, {df[num_col].isna().sum()} missing")

print(f"\nTotal columns after adding numeric versions: {len(df.columns)}")
print("New numeric columns added:")
for year in years:
    print(f"  emp_{year}_num")

Created emp_2017_num: 10264 valid values, 35821 missing
Created emp_2018_num: 17889 valid values, 28196 missing
Created emp_2019_num: 27824 valid values, 18261 missing
Created emp_2020_num: 25618 valid values, 20467 missing
Created emp_2021_num: 36159 valid values, 9926 missing
Created emp_2022_num: 36074 valid values, 10011 missing
Created emp_2023_num: 35867 valid values, 10218 missing
Created emp_2024_num: 31072 valid values, 15013 missing

Total columns after adding numeric versions: 27
New numeric columns added:
  emp_2017_num
  emp_2018_num
  emp_2019_num
  emp_2020_num
  emp_2021_num
  emp_2022_num
  emp_2023_num
  emp_2024_num


## Data Cleaning Summary

### Structure Changes:
- **Dropped columns**: `Unnamed: 0`
- **Renamed columns**: All columns given simple, consistent names
- **Regions**: Already standardized (9 Austrian states + Vienna)
- **Status**: Already in English and standardized

### Employee Data Structure:
**Raw columns** (emp_2017_raw to emp_2024_raw):
- Preserve original values including "n.a." strings
- Maintain data integrity and transparency

**Numeric columns** (emp_2017_num to emp_2024_num):
- Convert valid numbers to float
- Convert "n.a." to NaN (pandas missing) - **Rule 1**: If input unavailable, output unavailable
- Ready for mathematical operations while preserving missingness logic

### Missing Data Preservation:
- No imputation or guessing of missing values
- "n.a." strings preserved in raw columns
- Proper NaN handling in numeric columns
- Missingness patterns maintained for analysis

### Final Dataset:
- **Shape**: 46,085 rows × 27 columns
- **Raw employee columns**: 8 (2017-2024)
- **Numeric employee columns**: 8 (2017-2024)
- **Other columns**: 11 (company info, classification, etc.)

In [10]:
# Save cleaned dataset for next steps
output_path = '../data/processed/austria_cleaned.pkl'
df.to_pickle(output_path)
print(f"Cleaned dataset saved to: {output_path}")
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")

Cleaned dataset saved to: ../data/processed/austria_cleaned.pkl
Shape: (46085, 27)
Columns: 27
